# V2 TabM-mini challenger: frozen chronological audit

**Answer first.** The challenger is viable, and the pre-frozen
60% TabM / 40% capacity-LightGBM blend passed the one-shot Dev3
gate. This notebook is a read-only, executed companion to the
machine-readable artifacts. It does not train, tune, or write a
submission.

Dev1 and Dev2 select the blend. Dev3 confirms it once. Months
59-70 remain outside this notebook.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display

root = Path.cwd()
tabm_dir = root / 'artifacts' / 'v2' / 'tabm_mini'
diag_dir = root / 'artifacts' / 'v2' / 'diagnostics'

def load_json(path):
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest().upper()

summaries = {
    fold: load_json(tabm_dir / f'{fold.lower()}_summary.json')
    for fold in ('Dev1', 'Dev2', 'Dev3')
}
audit = load_json(diag_dir / 'tabm_capacity_frozen_dev3_audit.json')
pooled = load_json(diag_dir / 'tabm_capacity_pooled_report.json')
comparison = load_json(diag_dir / 'tabm_capacity_blend_comparison.json')
grid = pd.read_csv(diag_dir / 'tabm_capacity_screen_grid.csv')
monthly = pd.read_csv(diag_dir / 'tabm_capacity_monthly_scores.csv')

## Standalone TabM outer-fold results

In [ ]:
fold_table = pd.DataFrame([
    {
        'fold': fold,
        'outer train': f"{summary['outer_train_months'][0]}-{summary['outer_train_months'][1]}",
        'outer validation': f"{summary['outer_validation_months'][0]}-{summary['outer_validation_months'][1]}",
        'selected epoch': summary['best_epoch'],
        'inner cosine': summary['best_inner_ensemble_cosine'],
        'outer raw cosine': summary['outer_ensemble_cosine_raw'],
        'member mean cosine': summary['outer_member_cosine_mean'],
        'member pair correlation': summary['outer_member_mean_pairwise_correlation'],
        'elapsed seconds': sum(summary[key] for key in (
            'selection_transform_seconds', 'selection_training_seconds',
            'refit_transform_seconds', 'refit_training_seconds')),
    }
    for fold, summary in summaries.items()
])
display(fold_table)

Every outer score above is produced after discarding the inner
checkpoint and refitting from scratch on the full outer-training
window. Memberwise MSE is used during training; member predictions
are averaged only at inference.

## Dev1+2 selection screen

In [ ]:
display(grid.head(10))
selected = grid.loc[grid['selected']].iloc[0]
assert selected['tabm_weight'] == 0.6
assert selected['power_exponent'] == 1.1
assert abs(selected['screen_cosine'] - 0.14445559865492444) < 1e-14

## Frozen Dev3 audit and pooled reporting

In [ ]:
result_table = pd.DataFrame([
    {'view': 'Dev1+2 screen', 'cosine': pooled['screen_dev1_dev2_cosine'],
     'role': 'selection'},
    {'view': 'Dev3', 'cosine': audit['dev3_cosine'],
     'role': 'one-shot confirmation'},
    {'view': 'Dev1+2+Dev3 pooled',
     'cosine': pooled['pooled_dev1_dev2_dev3_cosine'],
     'role': 'reporting only'},
])
display(result_table)
display(pd.DataFrame([{
    'status': audit['status'],
    'positive-month share': audit['dev3_positive_month_share'],
    'gate passed': audit['promotion_gate']['passed'],
    'final TabM epoch': pooled['final_tabm_refit_epoch'],
    'frozen config SHA-256': audit['frozen_config_sha256'],
    'prediction SHA-256': audit['prediction_artifact_sha256'],
}]))

## Monthly robustness diagnostic

In [ ]:
monthly_summary = monthly.groupby('split', sort=False).agg(
    months=('month', 'count'),
    median_cosine=('cosine', 'median'),
    worst_cosine=('cosine', 'min'),
    positive_month_share=('positive_cosine', 'mean'),
).reset_index()
display(monthly_summary)
display(monthly[['split', 'month', 'n_rows', 'cosine']])

## Integrity replay

In [ ]:
prediction_path = diag_dir / 'tabm_capacity_frozen_dev3_predictions.npz'
assert audit['status'] == 'complete_one_shot_dev3_audit'
assert audit['selection_hash_checks_all_passed']
assert audit['promotion_gate']['passed']
assert audit['dev3_positive_month_share'] == 1.0
assert sha256_file(prediction_path) == audit['prediction_artifact_sha256']

with np.load(prediction_path, allow_pickle=False) as artifact:
    assert {'row_indices', 'months', 'target', 'prediction'} <= set(artifact.files)
    target = artifact['target'].astype(np.float64)
    prediction = artifact['prediction'].astype(np.float64)
    replay_cosine = float(
        np.dot(target, prediction)
        / (np.linalg.norm(target) * np.linalg.norm(prediction))
    )
    assert np.isfinite(prediction).all()
    assert abs(np.sqrt(np.mean(prediction ** 2)) - 1.0) < 1e-12
    assert abs(replay_cosine - audit['dev3_cosine']) < 1e-14

pd.DataFrame([{
    'check': 'prediction artifact hash, schema, RMS, and cosine replay',
    'passed': True,
    'replayed Dev3 cosine': replay_cosine,
}])

## Conclusion

The complementary neural component is real: standalone TabM beat
capacity LightGBM on the untouched Dev3 block, and the frozen blend
improved further to 0.1469999469 while retaining positive cosine in
all Dev3 months. The deployable recipe is therefore the frozen blend
with a six-epoch full-data TabM refit. Sealed-audit behavior remains
a separate safety-veto question and is not inspected here.